# Pneumonia Detection from Chest X-rays — CNN (Kaggle GPU)

Binary classification of paediatric chest radiographs: **normal** vs
**pneumonia**.

### Setup on Kaggle
1. **Add Input** -> search `chest-xray-pneumonia` -> add
   `paultimothymooney/chest-xray-pneumonia`
2. **Settings -> Accelerator -> GPU T4 x2**
3. **Run All** (roughly 10-15 minutes; 5,863 images)

### Two traps in this dataset, both handled below
- **The official `val` split has 16 images.** Eight per class is far too few to
  choose a stopping epoch from — validation accuracy would jump in 12.5% steps.
  A proper validation split is carved out of `train` instead.
- **The classes are 3:1 imbalanced** (3,875 pneumonia vs 1,341 normal). Left
  alone the network can score 74% by answering "pneumonia" every time, so class
  weights are applied and **recall on the normal class** is watched rather than
  raw accuracy.

In [ ]:
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus if gpus else "NONE - set Settings > Accelerator > GPU T4 x2")

In [ ]:
BASE = "/kaggle/input/chest-xray-pneumonia/chest_xray"
IMG_SIZE = 224
BATCH = 32
CLASSES = ["NORMAL", "PNEUMONIA"]

for split in ("train", "val", "test"):
    counts = {c: len(os.listdir(f"{BASE}/{split}/{c}")) for c in CLASSES}
    print(f"{split:<6} {counts}  total {sum(counts.values())}")

## 1. Datasets

Note the validation split comes out of `train`, not from the supplied `val`
folder — see the trap described above.

In [ ]:
common = dict(image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
              label_mode="int", class_names=CLASSES)

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/train", validation_split=0.15, subset="training", seed=SEED, **common)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/train", validation_split=0.15, subset="validation", seed=SEED, **common)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{BASE}/test", shuffle=False, **common)

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
n_normal = len(os.listdir(f"{BASE}/train/NORMAL"))
n_pneu = len(os.listdir(f"{BASE}/train/PNEUMONIA"))
total = n_normal + n_pneu
class_weight = {0: total / (2 * n_normal), 1: total / (2 * n_pneu)}
print(f"NORMAL {n_normal}  PNEUMONIA {n_pneu}")
print("class weights:", {k: round(v, 3) for k, v in class_weight.items()})
print(f"always-pneumonia baseline accuracy: {n_pneu / total:.2%}")

In [ ]:
plt.figure(figsize=(13, 7))
for images, labels in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(CLASSES[int(labels[i])], fontsize=11)
        plt.axis("off")
plt.suptitle("Chest X-ray training samples", fontsize=14, weight="bold")
plt.tight_layout(); plt.show()

## 2. Model

One sigmoid output; AUC and recall are tracked alongside accuracy.

In [ ]:
def build_model(img_size=IMG_SIZE):
    augment = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.06),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ], name="augment")

    base = tf.keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = augment(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs, outputs), base

model, base = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.Recall(name="recall")])
model.summary()

## 3. Stage 1 — frozen backbone

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=3, min_lr=1e-6, verbose=1),
]
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                  class_weight=class_weight, callbacks=callbacks)

## 4. Stage 2 — fine-tuning

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Recall(name="recall")])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=10,
                  class_weight=class_weight, callbacks=callbacks)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
off = 0
for h, lab in zip([hist1, hist2], ["frozen", "fine-tune"]):
    e = range(off, off + len(h.history["loss"]))
    for j, key in enumerate(["accuracy", "auc", "loss"]):
        ax[j].plot(e, h.history[key], label=f"{lab} train")
        ax[j].plot(e, h.history[f"val_{key}"], "--", label=f"{lab} val")
        ax[j].set_title(key); ax[j].set_xlabel("epoch"); ax[j].grid(alpha=.3)
    off += len(h.history["loss"])
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Evaluate

For a screening tool, **recall on pneumonia** (how many sick patients we catch)
matters more than overall accuracy. The threshold sweep below shows the
trade-off explicitly rather than silently assuming 0.5 is right.

In [ ]:
results = model.evaluate(test_ds, verbose=0)
print(dict(zip(model.metrics_names, [round(float(v), 4) for v in results])))

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(f"\nROC-AUC: {roc_auc_score(y_true, y_prob):.4f}\n")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, cbar=False)
plt.title("Confusion matrix (threshold 0.5)", weight="bold")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

In [ ]:
print(f"{'threshold':>10}{'accuracy':>10}{'precision':>11}{'recall':>9}{'missed':>8}")
print("-" * 48)
for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    p = (y_prob >= t).astype(int)
    tp = ((p == 1) & (y_true == 1)).sum()
    fp = ((p == 1) & (y_true == 0)).sum()
    fn = ((p == 0) & (y_true == 1)).sum()
    acc = (p == y_true).mean()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    print(f"{t:>10.1f}{acc:>10.4f}{prec:>11.4f}{rec:>9.4f}{fn:>8}")
print("\n'missed' = pneumonia cases called normal — the costly error here.")

## 6. Save

In [ ]:
model.save("/kaggle/working/pneumonia.keras")
metrics = {
    "task": "pneumonia",
    "classes": CLASSES,
    "test_accuracy": float((y_pred == y_true).mean()),
    "test_roc_auc": float(roc_auc_score(y_true, y_prob)),
    "confusion_matrix": cm.tolist(),
    "class_weights": {str(k): float(v) for k, v in class_weight.items()},
    "classification_report": classification_report(
        y_true, y_pred, target_names=CLASSES, output_dict=True),
}
with open("/kaggle/working/pneumonia_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("saved:", os.listdir("/kaggle/working"))

---

Download `pneumonia.keras` into the repo's `models/image/` folder and
`pneumonia_metrics.json` into `reports/`.

To wire it into the Streamlit app, add an entry to `IMAGE_TASKS` in
`src/config.py`:

```python
"pneumonia": {
    "name": "Pneumonia (Chest X-ray)",
    "icon": "🫁",
    "dir": IMAGES_DIR / "pneumonia",
    "train_subdir": "train", "test_subdir": "test",
    "classes": ["NORMAL", "PNEUMONIA"],
    "pretty_classes": ["Normal", "Pneumonia"],
    "img_size": 224,
    "source": "Kermany chest X-ray dataset",
    "blurb": "Binary classification of paediatric chest radiographs.",
},
```